# Falcon 40B

https://huggingface.co/tiiuae/falcon-40b

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import transformers
import torch
from accelerate import infer_auto_device_map, init_empty_weights

model_name = "tiiuae/falcon-7b"

In [17]:
config = AutoConfig.from_pretrained(model_name)
with init_empty_weights():
    model_causal_lm = AutoModelForCausalLM.from_config(config)

device_map = infer_auto_device_map(model_causal_lm, no_split_module_classes=["OPTDecoderLayer"])
print(device_map)

config.json: 100%|██████████| 1.05k/1.05k [00:00<00:00, 1.05MB/s]
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


{'transformer.word_embeddings': 0, 'transformer.h.0': 0, 'transformer.h.1': 0, 'transformer.h.2': 0, 'transformer.h.3': 0, 'transformer.h.4': 0, 'transformer.h.5': 0, 'transformer.h.6': 0, 'transformer.h.7': 0, 'transformer.h.8': 0, 'transformer.h.9': 0, 'transformer.h.10': 0, 'transformer.h.11.self_attention': 0, 'transformer.h.11.input_layernorm': 'cpu', 'transformer.h.12': 'cpu', 'transformer.h.13': 'cpu', 'transformer.h.14': 'cpu', 'transformer.h.15': 'cpu', 'transformer.h.16': 'cpu', 'transformer.h.17': 'cpu', 'transformer.h.18': 'cpu', 'transformer.h.19': 'cpu', 'transformer.h.20': 'cpu', 'transformer.h.21': 'cpu', 'transformer.h.22': 'cpu', 'transformer.h.23': 'cpu', 'transformer.h.24': 'cpu', 'transformer.h.25': 'cpu', 'transformer.h.26': 'cpu', 'transformer.h.27': 'cpu', 'transformer.h.28': 'cpu', 'transformer.h.29': 'cpu', 'transformer.h.30': 'cpu', 'transformer.h.31': 'cpu', 'transformer.ln_f': 'cpu', 'lm_head': 'cpu', 'transformer.h.11.mlp': 'cpu'}


In [20]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
    offload_folder="offload",
)
sequences = pipeline(
   "Girafatron is obsessed with giraffes, the most glorious animal on the face of this Earth. Giraftron believes all other animals are irrelevant when compared to the glorious majesty of the giraffe.\nDaniel: Hello, Girafatron!\nGirafatron:",
    max_length=200,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")

Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.82s/it]


ValueError: The following `model_kwargs` are not used by the model: ['offload_folder'] (note: typos in the generate arguments will also show up in this list)

In [3]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"]="1,2"
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import transformers
import torch

model_id = "tiiuae/falcon-40b"  # -instruct

tokenizer = AutoTokenizer.from_pretrained(model_id)
streamer = TextStreamer(tokenizer)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    # --- Choosing between 4, 8, and 16 bit --- #
    # 8bit: ~50GB GPU memory, fastest
    # 4bit: ~25GB GPU memory, slowest 
    # 16bit: ~100GB GPU memory, slow
    load_in_8bit=True, # torch_dtype=torch.bfloat16 or load_in_4bit=True
    trust_remote_code=True,
    device_map="auto",
)

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

ImportError: Using `load_in_8bit=True` requires Accelerate: `pip install accelerate` and the latest version of bitsandbytes `pip install -i https://test.pypi.org/simple/ bitsandbytes` or `pip install bitsandbytes`.

# END